# DEST — Anexo #2: Barrido alpha 0.1/0.3/0.5/0.7/0.9 (STANDALONE)

In [ ]:
# 0. Setup standalone
import os, sys, subprocess
print("🔧 Setup...")
if os.path.exists("DEST"): subprocess.call(["rm","-rf","DEST"])
subprocess.check_call(["git","clone","https://github.com/starlyn2010/DEST.git"])
subprocess.check_call([sys.executable,"-m","pip","install","-e","DEST","-q"])
if "DEST/src" not in sys.path: sys.path.insert(0,"DEST/src")
import dest
sys.modules["dest_lib"]=dest
for sub in ["config","samplers","models","datasets","runner"]:
    try: m=__import__(f"dest.{sub}", fromlist=[sub]); sys.modules[f"dest_lib.{sub}"]=m
    except: pass
print("✅ DEST instalado (fix Collatz 98% dup → 45k únicos)")
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# 1. Config Barrido alpha
from dest_lib.config import get_config
config=get_config("PAPER")
config["datasets"]=["CIFAR10"]
config["samplers"]=["collatz_sweep"]  # usa CollatzSweepSampler con alpha fijo
config["seeds"]=[200,201,202]
config["epochs"]=15
config["batch_size"]=128
config["lr"]=0.01
config["lr_schedule"]="cosine"
config["output_dir"]="./dest_barrido_alpha"
config["val_fraction"]=0.1
config["verbose"]=True
# Alphas a probar
alphas=[0.1,0.3,0.5,0.7,0.9]
print(f"Alphas: {alphas} × {len(config['seeds'])} seeds = {len(alphas)*len(config['seeds'])} runs")


In [ ]:
# 2. Ejecutar barrido (alpha × seed)
import os, time, json
from dest_lib.runner import ExperimentRunner
runner=ExperimentRunner(config)
total=len(alphas)*len(config["seeds"])
done=0
for alpha in alphas:
    for seed in config["seeds"]:
        exp_id=f"CIFAR10_alpha{alpha}"
        out_file=os.path.join(config["output_dir"], f"{exp_id}_collatz_sweep_seed_{seed}.json")
        if os.path.exists(out_file):
            try:
                j=json.load(open(out_file))
                if j.get("status")=="COMPLETE" and len(j.get("test_accs",[]))==15:
                    print(f"⏭️ alpha {alpha} seed {seed} ya completo"); done+=1; continue
                else: os.remove(out_file)
            except: os.remove(out_file) if os.path.exists(out_file) else None
        print(f"\n[{done+1}/{total}] alpha={alpha} seed {seed}")
        r=runner.run_single_seed(exp_id=exp_id, sampler_name="collatz_sweep", seed=seed, dataset="CIFAR10", alpha_fixed=alpha)
        print(f"✅ alpha {alpha} seed {seed}: {r.final_test_acc:.2f}%")
        done+=1


In [ ]:
# 3. Resumen curva accuracy vs alpha
import glob, json, numpy as np
from collections import defaultdict
files=[f for f in glob.glob("dest_barrido_alpha/*.json") if "sampler_name" in json.load(open(f))]
print(f"JSONs: {len(files)}")
if files:
    groups=defaultdict(list)
    for f in files:
        j=json.load(open(f))
        # extraer alpha del combo (guardado en config_snapshot o exp_id)
        # usamos alpha del nombre o del sampler
        alpha=j["config_snapshot"].get("alpha_fixed", j["experiment_id"].split("alpha")[-1])
        try: alpha=float(alpha)
        except: alpha=str(alpha)
        groups[alpha].append(j["final_test_acc"])
    for alpha in sorted(groups, key=lambda x: float(x) if str(x).replace('.','',1).isdigit() else x):
        arr=groups[alpha]
        print(f"alpha {alpha}: {np.mean(arr):.2f} ±{np.std(arr,ddof=1):.2f} n={len(arr)}")
    import matplotlib.pyplot as plt
    xs=sorted(groups, key=lambda x: float(x))
    means=[np.mean(groups[x]) for x in xs]
    stds=[np.std(groups[x],ddof=1) for x in xs]
    plt.errorbar([float(x) for x in xs], means, yerr=stds, marker='o', capsize=4)
    plt.xlabel("alpha_end"); plt.ylabel("Test acc %"); plt.title("Barrido alpha (CIFAR-10, n=3)")
    plt.grid(alpha=0.3)
    plt.savefig("dest_barrido_alpha/curva_alpha.png", dpi=200, bbox_inches="tight")
    plt.show()


In [ ]:
# Zip y descarga
import shutil, os, glob, json
files=[f for f in glob.glob("dest_*/*.json") if "sampler_name" in json.load(open(f))]
print(f"JSONs válidos: {len(files)}")
shutil.make_archive("resultados_"+title_md.split()[1],"zip","dest_"+title_md.split()[1].lower() if "Barrido" in title_md else "dest_batch")
print("ZIP listo")
from google.colab import files; files.download(glob.glob("*.zip")[0])
